Dynamic Hybrid Siamese Network 

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.utils import Sequence
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Flatten, GlobalAveragePooling2D, Concatenate, Dropout
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# Set image size
IMAGE_SIZE = (299, 299)  # InceptionV3 default input size

# Load handcrafted features
features_df = pd.read_csv("../Dataset/Features/signature_features.csv")

# Extract filenames and labels
file_names = features_df["filename"].values
labels = features_df["label"].values  # 0 = Genuine, 1 = Forged
features = features_df.iloc[:, 2:].values  # Exclude filename and label

# Create image pairs (Original + Verification)
image_pairs = []
labels_list = []

for i in range(len(file_names)):
    for j in range(i + 1, len(file_names)):
        if file_names[i][:5] == file_names[j][:5]:  # Ensure same person (D0_XX, D1_XX)
            image_pairs.append((file_names[i], file_names[j]))  # (Original, Verification)
            labels_list.append(1 if labels[i] != labels[j] else 0)  # 1 = Forged, 0 = Genuine

# Ensure feature vectors match the number of image pairs
min_len = min(len(image_pairs), len(features))
image_pairs = image_pairs[:min_len]
labels_list = labels_list[:min_len]
feature_vectors = features[:min_len]

# Data Generator Class
class SignatureDataGenerator(Sequence):
    def __init__(self, image_pairs, labels, features, batch_size=32, shuffle=True):
        self.image_pairs = np.array(image_pairs)
        self.labels = np.array(labels)
        self.features = np.array(features)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.image_pairs))
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.image_pairs) / self.batch_size))

    def __getitem__(self, index):
        indices = self.indices[index * self.batch_size:(index + 1) * self.batch_size]
        batch_pairs = self.image_pairs[indices]
        batch_features = self.features[indices]
        batch_labels = self.labels[indices]

        # Load images
        original_images = np.array([self.load_image(pair[0]) for pair in batch_pairs])
        verification_images = np.array([self.load_image(pair[1]) for pair in batch_pairs])

        # Reshape handcrafted features
        batch_features = batch_features.reshape(batch_features.shape[0], batch_features.shape[1], 1)

        return (original_images, verification_images, batch_features), batch_labels


    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def load_image(self, filename):
        folder = "Genuine" if filename.startswith("T_") else "Forged"
        img_path = f"../Dataset/Processing/AugmentedDataset/{folder}/{filename}"
        img = load_img(img_path, target_size=IMAGE_SIZE)
        img = img_to_array(img)
        return tf.keras.applications.inception_v3.preprocess_input(img)

# Split data into train and test
split_ratio = int(0.8 * len(image_pairs))
train_pairs, test_pairs = image_pairs[:split_ratio], image_pairs[split_ratio:]
train_labels, test_labels = labels_list[:split_ratio], labels_list[split_ratio:]
train_features, test_features = feature_vectors[:split_ratio], feature_vectors[split_ratio:]

# Create generators
train_generator = SignatureDataGenerator(train_pairs, train_labels, train_features, batch_size=32)
test_generator = SignatureDataGenerator(test_pairs, test_labels, test_features, batch_size=32)

# Build CNN (InceptionV3) feature extractor
base_model = InceptionV3(weights=None, include_top=False, input_shape=(299, 299, 3))
cnn_output = GlobalAveragePooling2D()(base_model.output)
cnn_model = Model(inputs=base_model.input, outputs=cnn_output)

# Define Inputs
original_input = Input(shape=(299, 299, 3), name="original_input")
verification_input = Input(shape=(299, 299, 3), name="verification_input")
feature_input = Input(shape=(train_features.shape[1], 1), name="feature_input")

# Pass images through CNN
original_embedding = cnn_model(original_input)
verification_embedding = cnn_model(verification_input)

# LSTM for handcrafted features
lstm_layer = LSTM(128, return_sequences=False)(feature_input)

# Merge embeddings
merged = Concatenate()([original_embedding, verification_embedding, lstm_layer])
dense_layer = Dense(128, activation='relu')(merged)
dropout_layer = Dropout(0.5)(dense_layer)
output_layer = Dense(1, activation='sigmoid')(dropout_layer)

# Define Siamese model
model = Model(inputs=[original_input, verification_input, feature_input], outputs=output_layer)

# Compile model
model.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])

# Train model using generators
model.fit(train_generator, validation_data=test_generator, epochs=25)

# Save trained model
model.save("../Model/signature_forgery_siamese_model.h5")
print("Model training completed and saved!")

# Prediction & Evaluation
y_pred_probs = model.predict(test_generator)
y_pred = (y_pred_probs > 0.5).astype(int)  # Convert probabilities to binary predictions
y_true = np.array(test_labels[:len(y_pred)])  # Ensure same length as predictions

# Compute evaluation metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
conf_matrix = confusion_matrix(y_true, y_pred)

# Print evaluation results
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")
print("Confusion Matrix:")
print(conf_matrix)


In [ ]:
# Evaluation
y_pred_probs = model.predict(test_generator)
y_pred = (y_pred_probs > 0.5).astype(int)  # Convert probabilities to binary predictions
y_true = np.array(test_labels[:len(y_pred)])  # Ensure correct length

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
conf_matrix = confusion_matrix(y_true, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")
print("Confusion Matrix:")
print(conf_matrix)